# Clase 3 · 02 · Clave-valor: perfiles de cliente al estilo Redis

En un modelo clave-valor la base sólo conoce dos cosas: una **clave** única y un **valor** opaco. Las operaciones son `GET`, `SET` y `DEL` por clave. A cambio de esa simplicidad, la búsqueda por clave cuesta O(1) y se reparte fácilmente entre nodos.

Caso de uso: cuando un cliente paga, el sistema antifraude necesita su perfil (país, cantidad de compras, último dispositivo) en milisegundos.

| Redis | Lo que hacemos acá |
|---|---|
| `SET customer:42 '{...}'` | Una fila `(key, value)` en la tabla Delta `nosql_kv_customer_profile` |
| `GET customer:42` | Filtro por `key` con Spark, y luego un `dict` de Python en memoria |
| Cluster con hash slots | Repartir claves entre nodos con una función hash |

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala")

In [ ]:
%run ../common/config

In [ ]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)

customers = qualified_table(config, "silver_customers")
products = qualified_table(config, "silver_products")
transactions = qualified_table(config, "silver_transactions")
for table in [customers, products, transactions]:
    assert spark.catalog.tableExists(table), f"Falta {table}. Ejecutá 00_preflight (y la clase 2)."
lab_path = f"{config.volume_path}/nosql_lab"
print(f"Esquema: {config.catalog}.{config.schema} · escala {config.scale}")

## 1. Construir el almacén clave-valor

El valor es un JSON serializado. Para la base es un texto: no sabe qué campos tiene.

In [ ]:
import json
import time

kv_table = qualified_table(config, "nosql_kv_customer_profile")

profile = (spark.table(transactions)
    .groupBy("customer_id")
    .agg(F.count("*").alias("tx_count"),
         F.round(F.sum("amount").cast("double"), 2).alias("total_amount"),
         F.sum("is_fraud").alias("fraud_transactions"),
         F.max_by("device_id", "event_ts").alias("last_device"),
         F.max("event_ts").cast("string").alias("last_seen")))

kv = (spark.table(customers).join(profile, "customer_id", "left")
    .select(F.concat(F.lit("customer:"), F.col("customer_id")).alias("key"),
            F.to_json(F.struct("country", "segment", "email",
                               F.coalesce("tx_count", F.lit(0)).alias("tx_count"),
                               "total_amount", "fraud_transactions", "last_device", "last_seen")).alias("value")))

kv.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(kv_table)
display(spark.table(kv_table).orderBy("key").limit(5))
print(f"{spark.table(kv_table).count():,} claves en {kv_table}")

## 2. `GET` por clave vs búsqueda por valor

Buscar por clave es directo. Pero si queremos *"todos los clientes de AR"*, el país está **dentro** del valor: hay que leer y parsear cada valor. En Redis no existe `WHERE country = 'AR'`; habría que mantener a mano una estructura extra, por ejemplo un set `country:AR` con las claves.

In [ ]:
def spark_get(key):
    row = spark.table(kv_table).where(F.col("key") == key).select("value").first()
    return json.loads(row.value) if row else None

print("GET customer:42 →", spark_get("customer:42"))

value_schema = ("country STRING, segment STRING, email STRING, tx_count BIGINT, total_amount DOUBLE, "
                "fraud_transactions BIGINT, last_device STRING, last_seen STRING")
by_country = (spark.table(kv_table)
    .select("key", F.from_json("value", value_schema).alias("v"))     # parsea TODOS los valores
    .where("v.country = 'AR'"))
print(f"Clientes de AR: {by_country.count():,} (se parsearon {spark.table(kv_table).count():,} valores)")

## 3. ¿Por qué Redis vive en memoria?

Cargamos todos los pares en un `dict` de Python, que es una **tabla hash en memoria**, igual que el corazón de Redis. Comparamos el tiempo de un `GET` en el `dict` con un `GET` mediante una consulta Spark sobre Delta.

In [ ]:
store = {row.key: row.value for row in spark.table(kv_table).collect()}   # nuestro "Redis"
rng = np.random.default_rng(42)
sample_keys = [str(k) for k in rng.choice(list(store.keys()), size=10_000)]

start = time.perf_counter()
for key in sample_keys:
    _ = store.get(key)
dict_ms = (time.perf_counter() - start) / len(sample_keys) * 1000

spark_times = []
for key in sample_keys[:5]:
    start = time.perf_counter()
    spark_get(key)
    spark_times.append((time.perf_counter() - start) * 1000)
spark_ms = float(np.median(spark_times))

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.barh(["GET en memoria (dict ≈ Redis)", "GET con Spark sobre Delta"], [dict_ms, spark_ms], color=["#16a34a", "#2563eb"])
ax.set_xscale("log")
ax.set_xlabel("Milisegundos por GET (escala logarítmica)")
ax.set_title(f"El dict es ~{spark_ms / dict_ms:,.0f} veces más rápido")
for i, v in enumerate([dict_ms, spark_ms]):
    ax.text(v, i, f"  {v:.4f} ms" if v < 1 else f"  {v:,.0f} ms", va="center")
plt.show()

Un Redis real agrega la ida y vuelta por red (≈0,1–1 ms), pero sigue siendo órdenes de magnitud más rápido que lanzar una consulta analítica. Spark está optimizado para **recorrer muchos datos**; un almacén clave-valor, para **encontrar uno** enseguida.

## 4. Repartir claves entre nodos con una función hash

Con muchos datos, las claves se reparten entre nodos: `nodo = hash(clave) mod N`. Una buena función hash reparte de forma pareja aunque las claves sean consecutivas.

In [ ]:
import hashlib
import bisect

def h(text):
    return int(hashlib.md5(text.encode()).hexdigest()[:8], 16)

keys = sorted(store.keys())

def modulo_node(key, n_nodes):
    return h(key) % n_nodes

load = pd.Series([modulo_node(k, 4) for k in keys]).value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar([f"nodo {i}" for i in load.index], load.values, color="#2563eb")
ax.axhline(len(keys) / 4, color="black", ls="--", lw=1, label="reparto ideal")
ax.set_title("hash(clave) mod 4: cada nodo recibe ≈25% de las claves")
ax.set_ylabel("Claves")
ax.legend()
plt.show()

### ¿Qué pasa si agregamos un nodo?

Con `mod N`, pasar de 4 a 5 nodos cambia el resultado de casi todas las claves: hay que **mover** la mayoría de los datos por la red. El **hashing consistente** (usado por Dynamo y Cassandra) ubica nodos y claves sobre un anillo: al agregar un nodo sólo se mueven las claves del tramo que ocupa. Los **nodos virtuales** (varias posiciones por nodo) equilibran la carga.

In [ ]:
class HashRing:
    def __init__(self, nodes, vnodes=100):
        self.points = sorted((h(f"{node}#{i}"), node) for node in nodes for i in range(vnodes))
        self.hashes = [p for p, _ in self.points]

    def node_for(self, key):
        i = bisect.bisect(self.hashes, h(key)) % len(self.points)
        return self.points[i][1]

def moved_fraction(before, after):
    return sum(before[k] != after[k] for k in keys) / len(keys)

def experiment(n_from=4, n_to=5, vnodes=100):
    mod_before = {k: modulo_node(k, n_from) for k in keys}
    mod_after = {k: modulo_node(k, n_to) for k in keys}
    ring_before, ring_after = HashRing(range(n_from), vnodes), HashRing(range(n_to), vnodes)
    rb = {k: ring_before.node_for(k) for k in keys}
    ra = {k: ring_after.node_for(k) for k in keys}
    loads = pd.Series(list(ra.values())).value_counts()
    return {"de": n_from, "a": n_to, "vnodes": vnodes,
            "movidas_mod_%": 100 * moved_fraction(mod_before, mod_after),
            "movidas_anillo_%": 100 * moved_fraction(rb, ra),
            "carga_max_anillo_%": 100 * loads.max() / len(keys)}

result = experiment(4, 5, vnodes=100)
display(pd.DataFrame([result]).round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].bar(["hash mod N", "hashing consistente"], [result["movidas_mod_%"], result["movidas_anillo_%"]], color=["#dc2626", "#16a34a"])
axes[0].set_ylabel("% de claves que cambian de nodo")
axes[0].set_title("Agregar un nodo: de 4 a 5")
for vnodes, color in [(1, "#f59e0b"), (100, "#16a34a")]:
    ring = HashRing(range(5), vnodes)
    share = pd.Series([ring.node_for(k) for k in keys]).value_counts().sort_index() / len(keys) * 100
    axes[1].plot([f"nodo {i}" for i in share.index], share.values, marker="o", color=color, label=f"{vnodes} nodo(s) virtual(es)")
axes[1].axhline(20, color="black", ls="--", lw=1, label="ideal 20%")
axes[1].set_ylabel("% de claves por nodo")
axes[1].set_title("Carga por nodo en el anillo (5 nodos)")
axes[1].legend()
fig.tight_layout()
plt.show()

## Distintos tamaños de cluster y nodos virtuales

Repetimos el experimento con varias configuraciones. Fijate que con `mod N` se mueve casi todo, mientras que con el anillo se mueve aproximadamente `1/(N+1)` de las claves, y que con pocos nodos virtuales la carga queda desbalanceada.

In [ ]:
cases = [(4, 5, 1), (4, 5, 10), (4, 5, 100), (10, 11, 100)]
display(pd.DataFrame([experiment(a, b, v) for a, b, v in cases]).round(1))